# Experiment 2: Data Cleaning

**Aim:** Handle missing values, rename columns, change types and calculate a new field.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default='notebook_connected'
DATA=Path('data') if Path('data').exists() else Path('DV/data')
df=pd.read_csv(DATA/'retail_sales.csv',parse_dates=['Date'])

raw=pd.read_csv(DATA/'retail_dirty.csv')
print('Before:',raw.isna().sum().to_dict())
clean=raw.rename(columns={'Order ID':'OrderID','Sales Amount':'Sales'}).drop_duplicates().copy()
clean['Date']=pd.to_datetime(clean.Date)
for c in ['Sales','Profit','Quantity']:
    clean[c]=pd.to_numeric(clean[c],errors='coerce')
    clean[c]=clean[c].fillna(clean[c].median())
clean['Category']=clean.Category.str.strip()
clean['ProfitMargin']=clean.Profit/clean.Sales
print('After:',clean.isna().sum().to_dict())
print(clean.head().to_string(index=False))

Before: {'Order ID': 0, 'Date': 0, 'Region': 0, 'Country': 0, 'State': 0, 'City': 0, 'Latitude': 0, 'Longitude': 0, 'Category': 0, 'Subcategory': 0, 'Quantity': 1, 'Sales Amount': 1, 'Profit': 1}
After: {'OrderID': 0, 'Date': 0, 'Region': 0, 'Country': 0, 'State': 0, 'City': 0, 'Latitude': 0, 'Longitude': 0, 'Category': 0, 'Subcategory': 0, 'Quantity': 0, 'Sales': 0, 'Profit': 0, 'ProfitMargin': 0}
OrderID       Date Region Country       State      City  Latitude  Longitude        Category Subcategory  Quantity    Sales  Profit  ProfitMargin
  O0001 2023-01-01  North   India       Delhi     Delhi   28.6139    77.2090 Office Supplies       Paper       6.0  664.550  131.95      0.198555
  O0002 2023-01-04  South   India   Karnataka Bengaluru   12.9716    77.5946      Technology Accessories       1.0  285.670   59.82      0.209402
  O0003 2023-01-07   East   India West Bengal   Kolkata   22.5726    88.3639 Office Supplies       Paper       5.0  985.975   70.45      0.071452
  O0004 2023-0

## Power BI Desktop procedure
1. Import retail_dirty.csv → Transform Data. Rename Order ID to OrderID and Sales Amount to Sales.
2. Remove duplicates, set Date to Date and Sales, Profit and Quantity to Decimal Number. Trim Category (Transform → Format → Trim).
3. Replace missing numeric values with medians. Example Power Query M formula (replace PreviousStep with actual preceding step):
   ```powerquery
   = Table.ReplaceValue(PreviousStep, null, List.Median(List.RemoveNulls(PreviousStep[Sales])), Replacer.ReplaceValue, {"Sales"})
   ```
   Repeat for Profit and Quantity.
4. Add Custom Column ProfitMargin with formula `if [Sales] = 0 then null else [Profit]/[Sales]`. Close & Apply.